In [ ]:
# Random Forest — FI prediction (calibration / training)
# Trains and evaluates a default Random Forest model for FI on the calibration
# dataset, using engineered features derived from the normalized logs.
# The step applying the trained model to the full 318-well log dataset is not
# included here; those logs are publicly available from the WOGCC.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

# Load calibration data (place the file in the same folder, or update the path)
file_path = 'The last final FI for modeling.xlsx'
df = pd.read_excel(file_path, sheet_name='Sheet1')
df = df.dropna(subset=['ResDeep_N', 'GR_N', 'RHOB_N', 'FI']).copy()

# Feature engineering
df['ResDeep_N_x_RHOB_N'] = df['ResDeep_N'] * df['RHOB_N']
df['RHOB_N_x_GR_N'] = df['RHOB_N'] * df['GR_N']
df['GR_N_sq'] = df['GR_N'] ** 2
df['ResDeep_N_sq'] = df['ResDeep_N'] ** 2
df['log_GR'] = np.log(df['GR_N'] + 1e-6)

feature_cols = [
    'ResDeep_N', 'GR_N', 'RHOB_N',
    'ResDeep_N_x_RHOB_N', 'RHOB_N_x_GR_N',
    'GR_N_sq', 'ResDeep_N_sq', 'log_GR'
]
X = df[feature_cols].values
y = df['FI'].values
FI_mean = np.mean(y)

# Default Random Forest fit
rf = RandomForestRegressor(random_state=42)
rf.fit(X, y)

# Calibration performance
y_pred = rf.predict(X)
r2 = r2_score(y, y_pred)
mae = mean_absolute_error(y, y_pred)
print(f"[Calibration] R2 = {r2:.4f} | MAE = {mae:.3f} | Mean FI = {FI_mean:.4f}")

# Predicted vs observed (calibration)
plt.figure(figsize=(7, 6))
plt.scatter(y, y_pred, color='darkblue', edgecolors='k', label='Samples')
plt.plot([min(y), max(y)], [min(y), max(y)], 'r--', label='45 deg reference')
plt.xlabel('Calculated FI')
plt.ylabel('Predicted FI')
plt.title('Predicted vs Calculated FI (Random Forest with engineered features)')
plt.legend(); plt.grid(True)
plt.text(0.025, 0.85,
         f"R2 = {r2:.4f}\nMAE = {mae:.3f}\nMean FI = {FI_mean:.4f}",
         transform=plt.gca().transAxes, fontsize=12, verticalalignment='top',
         bbox=dict(facecolor='white', alpha=0.7))
plt.tight_layout()
plt.show()

# 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
r2_scores, mae_scores = [], []

plt.figure(figsize=(15, 12))
for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    rf.fit(X_train, y_train)
    y_pred_fold = rf.predict(X_test)

    r2_fold = r2_score(y_test, y_pred_fold)
    mae_fold = mean_absolute_error(y_test, y_pred_fold)
    r2_scores.append(r2_fold)
    mae_scores.append(mae_fold)

    plt.subplot(3, 2, fold + 1)
    plt.scatter(y_test, y_pred_fold, edgecolors='k')
    plt.plot([min(y), max(y)], [min(y), max(y)], 'r--', label='45 deg reference')
    plt.xlabel('Calculated FI')
    plt.ylabel('Predicted FI')
    plt.title(f'Fold {fold + 1}: Predicted vs Calculated FI')
    plt.grid(True); plt.legend()
    plt.text(0.02, 0.95,
             f"Test MAE = {mae_fold:.3f}\nTest Mean = {np.mean(y_test):.3f}",
             transform=plt.gca().transAxes, fontsize=12, verticalalignment='top',
             bbox=dict(facecolor='white', alpha=0.7))

plt.tight_layout()
plt.suptitle('Cross-Validation: Predicted vs Calculated FI', fontsize=16, y=1.02)
plt.show()

print("\nCross-Validation Summary (test sets):")
print(f"MAE Mean +/- Std = {np.mean(mae_scores):.3f} +/- {np.std(mae_scores):.3f}")